In [1]:
from functools import cache

import pandas as pd
from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.sql import query_sql


@cache
def gcs_pandas():
    return GCSPandas()


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Get NTD ID from warehouse
for all California agences in NTD

In [2]:
# need to get ntd ID of all California agencies
# also include VOMs for comparison
# calling out specific key for 2024 Yuma County
ca_agencies = query_sql(
    """
    SELECT distinct 
        ntd_id, 
        agency, 
        agency_voms,
        report_year
    FROM cal-itp-data-infra.mart_ntd.dim_annual_service_agencies
    WHERE 
        state = 'CA' 
        AND report_year = 2024
        OR key = "49ae0c61e47b44a037c90d9d305bc416"
    """,
    as_df=True,
)
ca_agencies = ca_agencies.add_suffix("_mart")
ca_agencies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ntd_id_mart       210 non-null    object 
 1   agency_mart       210 non-null    object 
 2   agency_voms_mart  210 non-null    float64
 3   report_year_mart  210 non-null    int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 6.7+ KB


In [3]:
ca_agencies[ca_agencies["agency_mart"].str.contains("Yuma")] # confirmed Yuma County is here

,ntd_id_mart,agency_mart,agency_voms_mart,report_year_mart
59,90233,Yuma County Intergovernmental Public Transport...,55.0,2024


# Read in NTD 2024 Revenue Vehicle Inventory
this list all the revenue vehicles reported by each agency

In [4]:
# saved excel to GCS, so use gcs_pandas()
this_gcs_path = "gs://calitp-analytics-data/data-analyses/ntd/"

## NTD Glossary Definitions
- `Active Vehicles`
> The vehicles available to operate in revenue service at the end of the fiscal year, including: 
•   Spares
•   Vehicles temporarily out of service for routine maintenance and minor repairs
•   Operational vehicles

- Active Vehicles in Fleet
>The vehicles in a particular fleet at year-end that are available to operate in revenue service, including: 
•   Spares
•   Vehicles temporarily out of service for routine maintenance and minor repairs

- Number of Active Vehicles in Fleet
>The total number of operational revenue vehicles in the fleet available for general public transit service, including spare or back up revenue vehicles. The total should also include any operational revenue vehicles used by contractors in general public transit service. Non-revenue service vehicles and personal vehicles should not be included. Can be found in: A-30

- `Vehicles in Total Fleet`
>All revenue vehicles held at the end of the fiscal year, including those: 
•   In service;
•   In storage;
•   Emergency contingency; and
•   Awaiting sale.
Can be found in: A-30

- `Vehicles Operated in Annual Maximum Service (VOMS)`
>The number of revenue vehicles operated to meet the annual maximum service requirement. This is the revenue vehicle count during the peak season of the year; on the week and day that maximum service is provided. Vehicles operated in maximum service (VOMS) exclude: 
•   Atypical days; or
•   One-time special events.
Can be found in: B-30, A-30, S-10, Declarations, MR-20

In [5]:
rev_veh_inv_link = f"{this_gcs_path}2024 Revenue Vehicle Inventory_250820.xlsx"

rev_veh = gcs_pandas().read_excel(rev_veh_inv_link)

rev_veh.columns = rev_veh.columns.str.lower().str.strip().str.replace(" ", "_")
rev_veh["ntd_id"] = rev_veh["ntd_id"].astype("str")
rev_veh.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35218 entries, 0 to 35217
Data columns (total 41 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   state/parent_ntd_id                           13899 non-null  object 
 1   ntd_id                                        35218 non-null  object 
 2   agency_name                                   35218 non-null  object 
 3   reporter_type                                 35218 non-null  object 
 4   reporting_module                              35218 non-null  object 
 5   group_plan_sponsor_ntdid                      18739 non-null  object 
 6   group_plan_sponsor_name                       18739 non-null  object 
 7   modes                                         35218 non-null  object 
 8   revenue_vehicle_inventory_id                  35218 non-null  int64  
 9   agency_fleet_id                               35218 non-null 

In [6]:
# filter the NTD list by the NTD id from the warehouse

ca_rev_veh = rev_veh[
    rev_veh["ntd_id"].isin(ca_agencies["ntd_id_mart"].unique().tolist())
]

In [7]:
# group by / aggregate by active fleet, total fleet, VOMS.
# maybe there is a difference between all of the. I expect the toal fleet to be the highest

ca_rev_veh = (
    ca_rev_veh.groupby(
        [
            "ntd_id",
            "agency_name",
        ]
    )
    .agg(
        {
            "total_fleet_vehicles": "sum",
            "active_fleet_vehicles": "sum",
            "modetos_vehicles_operated_in_maximum_service": "max",
        }
    )
    .reset_index()
)

In [8]:
ca_rev_veh.head(3)

,ntd_id,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service
0,90003,San Francisco Bay Area Rapid Transit District,785,776,566.0
1,90004,Golden Empire Transit District,160,160,51.0
2,90006,Santa Cruz Metropolitan Transit District,147,147,58.0


## difference between VOMs and Rev Vehicle Inv.

In [9]:
diff_voms_ref_veh = ca_rev_veh.merge(
    ca_agencies,
    left_on="ntd_id",
    right_on="ntd_id_mart",
    how="inner",
    # suffixes = ("_ntd_excel","_warehouse"),
    # indicator=True # dont really need this since how=inner
)

In [10]:
diff_voms_ref_veh.head(3)

,ntd_id,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service,ntd_id_mart,agency_mart,agency_voms_mart,report_year_mart
0,90003,San Francisco Bay Area Rapid Transit District,785,776,566.0,90003,"San Francisco Bay Area Rapid Transit District,...",582.0,2024
1,90004,Golden Empire Transit District,160,160,51.0,90004,Golden Empire Transit District,93.0,2024
2,90006,Santa Cruz Metropolitan Transit District,147,147,58.0,90006,Santa Cruz Metropolitan Transit District,97.0,2024


## Calculate the differencs in VOMS values between the revenue vehicle inventory and the warehouse
Note, the warehouse table pull from a NTD excel file as well, so VOMS should be the same...

In [11]:
diff_voms_ref_veh["voms_diff"] = (
    diff_voms_ref_veh["agency_voms_mart"]
    - diff_voms_ref_veh["modetos_vehicles_operated_in_maximum_service"]
)

In [12]:
diff_voms_ref_veh["total_fleet_veh_voms_diff"] = (
    diff_voms_ref_veh["total_fleet_vehicles"]
    - diff_voms_ref_veh["modetos_vehicles_operated_in_maximum_service"]
)

In [13]:
diff_voms_ref_veh["ntd_id"].nunique()

210

## save data to GCS

In [14]:
diff_voms_ref_veh.to_csv(f"{this_gcs_path}1943_total_revenue_vehicles.csv")

## read in GCS data

In [15]:
diff_voms_ref_veh = gcs_pandas().read_csv(
    f"{this_gcs_path}1943_total_revenue_vehicles.csv"
)
diff_voms_ref_veh[["ntd_id", "ntd_id_mart"]] = diff_voms_ref_veh[
    ["ntd_id", "ntd_id_mart"]
].astype("str")

In [16]:
display(diff_voms_ref_veh.info(), diff_voms_ref_veh.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 12 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Unnamed: 0                                    210 non-null    int64  
 1   ntd_id                                        210 non-null    object 
 2   agency_name                                   210 non-null    object 
 3   total_fleet_vehicles                          210 non-null    int64  
 4   active_fleet_vehicles                         210 non-null    int64  
 5   modetos_vehicles_operated_in_maximum_service  210 non-null    float64
 6   ntd_id_mart                                   210 non-null    object 
 7   agency_mart                                   210 non-null    object 
 8   agency_voms_mart                              210 non-null    float64
 9   report_year_mart                              210 non-null    int

None

,Unnamed: 0,ntd_id,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service,ntd_id_mart,agency_mart,agency_voms_mart,report_year_mart,voms_diff,total_fleet_veh_voms_diff
0,0,90003,San Francisco Bay Area Rapid Transit District,785,776,566.0,90003,"San Francisco Bay Area Rapid Transit District,...",582.0,2024,16.0,219.0
1,1,90004,Golden Empire Transit District,160,160,51.0,90004,Golden Empire Transit District,93.0,2024,42.0,109.0
2,2,90006,Santa Cruz Metropolitan Transit District,147,147,58.0,90006,Santa Cruz Metropolitan Transit District,97.0,2024,39.0,89.0


# Count of routes

Eric used one of the new roll up tables using this query
```
SELECT DISTINCT month, month_first_day, schedule_name, n_routes, day_type
FROM `cal-itp-data-infra.mart_gtfs_rollup.fct_monthly_operator_summary`
WHERE month_first_day = '2026-02-01' AND n_routes > 0 AND day_type = 'Weekday'
ORDER BY n_routes DESC
```

In [17]:
# rollup_routes = gcs_pandas().read_csv(
#     f"{this_gcs_path}agency_routes_02-2026_rollup_routes_results.csv"
# )

# rollup_routes.info()

# this has shcedule name but not agency/analysis name. may need the custom query to join the bridge table to get agency name (ntd?)

In [18]:
# Query the bridge table, then query the monthly operator summary
# joing on gtfs schedule name

# Compton schedule is showing up in the rollup table, but not the bridge table...

query = """
WITH
  t1 AS (
    SELECT *
    FROM
      cal-itp-data-infra.mart_transit_database.bridge_gtfs_analysis_name_x_ntd
  ),
  t2 AS (
    SELECT DISTINCT month, month_first_day, schedule_name, n_routes, day_type
    FROM `cal-itp-data-infra.mart_gtfs_rollup.fct_monthly_operator_summary`
    WHERE
      month_first_day = '2026-02-01' AND n_routes > 0 AND day_type = 'Weekday'
    ORDER BY n_routes DESC
  )
SELECT DISTINCT
  t1.schedule_gtfs_dataset_name,
  t1.analysis_name,
  t1.ntd_id_2022,
  t2.schedule_name,
  t2.n_routes
FROM
  t1
inner JOIN t2
  ON t1.schedule_gtfs_dataset_name = t2.schedule_name;
"""
rollup_route_bridge = query_sql(query, as_df=True)

rollup_route_bridge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 5 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   schedule_gtfs_dataset_name  197 non-null    object
 1   analysis_name               197 non-null    object
 2   ntd_id_2022                 163 non-null    object
 3   schedule_name               197 non-null    object
 4   n_routes                    197 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 7.8+ KB


# Agencies without GTFS-RT
via CS team

In [19]:
no_rt = gcs_pandas().read_excel(
    f"{this_gcs_path}organizations_without_gtfs_rt_data_by_total_voms_2026-02-23_CS additions 1 (003).xlsx"
)

no_rt.columns = no_rt.columns.str.lower().str.strip().str.replace(" ", "_")
no_rt = no_rt[
    no_rt["organization_name"].notna()
]
no_rt["ntd_id"] = no_rt["ntd_id"].astype("int64").astype("str")


# update Plumas County to Plumas County Transportation Commission
no_rt.loc[no_rt["organization_name"]=="Plumas County","organization_name"] = "Plumas County Transportation Commission"

display(
    no_rt.info(),
    no_rt[no_rt["organization_name"].str.contains('Baldwin')]
)

<class 'pandas.core.frame.DataFrame'>
Index: 54 entries, 0 to 53
Data columns (total 9 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   organization_name                                 54 non-null     object 
 1   ntd_id                                            54 non-null     object 
 2   service_name                                      54 non-null     object 
 3   total_2023_voms                                   54 non-null     float64
 4   is_the_agency_pursuing_rt_(via_the_msas_or_not)?  10 non-null     object 
 5   notes                                             10 non-null     object 
 6   passiogo                                          3 non-null      object 
 7   gtfs_rt                                           50 non-null     object 
 8   if_gtfs-rt_data_-_add_the_link                    2 non-null      object 
dtypes: float64(1), object(8)
memo

None

,organization_name,ntd_id,service_name,total_2023_voms,is_the_agency_pursuing_rt_(via_the_msas_or_not)?,notes,passiogo,gtfs_rt,if_gtfs-rt_data_-_add_the_link
30,City of Baldwin Park,90251,Baldwin Park Express,8.0,NaN,NaN,NaN,yes,"Public Transit | Baldwin Park, CA"


# 1st attempt: Merge
1. `diff_voms_ref_veh` - to - `no_rt`
2. `rollup_routes` - to - `no_rt`

Hopefully the names are consistent between the datasets

## List of total Agency revenue vehicles, that do not have GTFS-RT

In [20]:
# no_rt_keep_cols = [
#     "organization_name",
#     "service_name",
#     # 'total_2023_voms',
#     "is_the_agency_pursuing_rt_(via_the_msas_or_not)?",
#     # 'notes',
#     # 'passiogo',
#     "gtfs_rt",
#     # 'if_gtfs-rt_data_-_add_the_link'
# ]
# no_rt_rev_veh_inv = diff_voms_ref_veh.merge(
#     no_rt[no_rt_keep_cols],
#     left_on="agency_name",
#     right_on="organization_name",
#     how="inner",
# )

# no_rt_rev_veh_inv

## List of total Agency routes, that do not have GTFS-RT

In [21]:
# no_rt_rollup_routes = rollup_route_bridge.merge(
#     no_rt[no_rt_keep_cols],
#     left_on="analysis_name",
#     right_on="organization_name",
#     how="inner",
# )

In [22]:
# display(no_rt_rollup_routes.info(), no_rt_rollup_routes)

# 2nd attempt; Merge total agency revenue vehicles to routes, THEN merge to agencies without gtfs-rt

1. `diff_voms_ref_veh` - to - `rollup_route_bridge`
2. above - to - `no_rt`

## merge revenue vehicle to rollup route summary

In [23]:
rev_veh_route = diff_voms_ref_veh.merge(
    rollup_route_bridge,
    left_on="ntd_id_mart",
    right_on="ntd_id_2022",
    how="inner",
)

rev_veh_route.info()  

# there are some unmerged agenies (77). this makes sense, not everyone in the NTD can be cross-walked to a gtfs schedule.
# also confirmed in meeting that some agencies are missing in the bridge table/air table 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 17 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Unnamed: 0                                    155 non-null    int64  
 1   ntd_id                                        155 non-null    object 
 2   agency_name                                   155 non-null    object 
 3   total_fleet_vehicles                          155 non-null    int64  
 4   active_fleet_vehicles                         155 non-null    int64  
 5   modetos_vehicles_operated_in_maximum_service  155 non-null    float64
 6   ntd_id_mart                                   155 non-null    object 
 7   agency_mart                                   155 non-null    object 
 8   agency_voms_mart                              155 non-null    float64
 9   report_year_mart                              155 non-null    int

## merge rev_veh_route to no_rt list

In [24]:
rev_veh_route_no_rt = rev_veh_route.merge(
    no_rt,
    left_on="ntd_id",
    right_on="ntd_id",
    how="inner",
    indicator=True,
)

rev_veh_route_no_rt_outer_merge = rev_veh_route.merge(
    no_rt,
    left_on="ntd_id",
    right_on="ntd_id",
    how="outer",
    indicator=True,
)

display(
    rev_veh_route_no_rt.info(),
    rev_veh_route_no_rt["_merge"].value_counts(),
    rev_veh_route_no_rt_outer_merge["_merge"].value_counts()
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 26 columns):
 #   Column                                            Non-Null Count  Dtype   
---  ------                                            --------------  -----   
 0   Unnamed: 0                                        51 non-null     int64   
 1   ntd_id                                            51 non-null     object  
 2   agency_name                                       51 non-null     object  
 3   total_fleet_vehicles                              51 non-null     int64   
 4   active_fleet_vehicles                             51 non-null     int64   
 5   modetos_vehicles_operated_in_maximum_service      51 non-null     float64 
 6   ntd_id_mart                                       51 non-null     object  
 7   agency_mart                                       51 non-null     object  
 8   agency_voms_mart                                  51 non-null     float64 
 9   report_year_

None

_merge
both          51
left_only      0
right_only     0
Name: count, dtype: int64

_merge
left_only     104
both           51
right_only      7
Name: count, dtype: int64

In [33]:
rev_veh_route_no_rt_outer_merge.shape

(162, 26)

In [36]:
rev_veh_route_no_rt_outer_merge[rev_veh_route_no_rt_outer_merge["_merge"]=="right_only"][["ntd_id","organization_name"]]

# City of Compton, City of Covina, City of El Segundo, City of San Fernando didnt matchbecause they arent in the bridge table?

,ntd_id,organization_name
89,90251,City of Baldwin Park
98,90260,City of Compton
99,90261,City of Covina
125,91008,Modoc Transportation Agency
155,99316,Chemehuevi Indian Tribe
159,99449,City of El Segundo
160,99451,City of San Fernando


In [26]:
drop_cols = [
    # "Unnamed: 0",
    # "ntd_id",
    "agency_name",
    "total_fleet_vehicles",
    "active_fleet_vehicles",
    "modetos_vehicles_operated_in_maximum_service",
    # "ntd_id_mart",
    # "agency_mart",
    # "agency_voms_mart",
    # "voms_diff",
    # "total_fleet_veh_voms_diff",
    "schedule_gtfs_dataset_name",
    "analysis_name",
    "ntd_id_2022",
    # "schedule_name",
    "n_routes",
    # "organization_name",
    "service_name",
    # "total_2023_voms",
    "is_the_agency_pursuing_rt_(via_the_msas_or_not)?",
    "notes",
    "passiogo",
    "gtfs_rt",
    "if_gtfs-rt_data_-_add_the_link",
    "_merge",
]
clean_rev_veh_route_no_rt = rev_veh_route_no_rt[drop_cols]

clean_rev_veh_route_no_rt_outer_merge = rev_veh_route_no_rt_outer_merge[drop_cols]

### save to GCS

In [27]:
clean_rev_veh_route_no_rt.to_csv(f"{this_gcs_path}1943_total_revenue_vehicles_routes_no_rt.csv")
clean_rev_veh_route_no_rt_outer_merge.to_csv(f"{this_gcs_path}1943_total_revenue_vehicles_routes_no_rt_outer_merge.csv")

### read in data

In [28]:
clean_check = gcs_pandas().read_csv(
    f"{this_gcs_path}1943_total_revenue_vehicles_routes_no_rt.csv"
)
clean_check_outer_merge = gcs_pandas().read_csv(
    f"{this_gcs_path}1943_total_revenue_vehicles_routes_no_rt_outer_merge.csv"
)

clean_check.head()

,Unnamed: 0,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service,schedule_gtfs_dataset_name,analysis_name,ntd_id_2022,n_routes,service_name,is_the_agency_pursuing_rt_(via_the_msas_or_not)?,notes,passiogo,gtfs_rt,if_gtfs-rt_data_-_add_the_link,_merge
0,0,Sacramento Regional Transit District,557,508,193.0,Sacramento Schedule,Sacramento Regional Transit District,90019,64,South County Transit Link,Maybe,"Might be onboarding using the MSAs, but progre...",NaN,No,NaN,both
1,1,City of Corona,20,20,6.0,Corona Schedule,City of Corona,90052,2,Corona Cruiser,Maybe,Expressed interest in RT through MSAs,NaN,No,NaN,both
2,2,City of Laguna Beach,32,32,18.0,Laguna Beach Schedule,City of Laguna Beach,90119,3,Laguna Beach Trolley,NaN,NaN,NaN,No,NaN,both
3,3,City of Lompoc,16,16,2.0,Lompoc Schedule,City of Lompoc,90149,6,City of Lompoc Transit,NaN,NaN,NaN,No,NaN,both
4,4,Los Angeles County Metropolitan Transportation...,4415,4231,1476.0,LA Metro Rail Schedule,Los Angeles County Metropolitan Transportation...,90154,6,Los Angeles County Transit Services,NaN,NaN,NaN,No,NaN,both


# Feedback

## Some agencies have missing numbers
why?

- `Yuma County Intergovernmental Public Transportation Authority` -> ~~Yuma Arizona?! 90233? AZ are filtered out.~~ FIXED, i added them manually. GTG
- `City of Alhambra` -> in rev vehicle and route list, but not part of the no_rt list, so was dropped in the final merge. GTG
- `City of Cerritos` -> in rev vehicle and route list, but not part of the no_rt list, so was dropped in the final merge. GTG
- `City of Baldwin Park` -> ~~in rev veh and routes list, but didnt merge because names were different.~~ Is in the no_rt list but is marked YES for having gtfs-RT. Not in bridge table so didnt make it to the routes list.
- `City of Calabasas` -> ~~in rev vehicle and route list, but not part of the no_rt list.~~ In the no_rt list, they have gtfs-rt. GTG
- `City of Compton` - > in rev veh and no_rt list. Not in bridge table so didnt make it to the routes list.
- `Plumas County` -> ~~is in the rev veh-to-routes merge and no_rt list, wasnt joining on the name. can fix this manually (plumas county vs plumas county transportation comission)~~ FIXED. 
- `City of Covina` - > in rev veh and no_rt list, Not in bridge table so didnt make it to the routes list.
- `Modoc Transportation Agency` ->  in rev vehicle list and no rt list. Not in bridge table so didnt make it to the routes list.
- `City of El Segundo` -> ntd id 99449 under LA Metro? not  in the rev vehicle or routes list, only in no_rt list. Reduced Asset Reporter in NTD
- `City of San Fernando` -> under LA Metro as well? in routes and no_rt list, not in rev veh list. Reduced Asset Reporter in NTD


In [34]:
agency = "Chemehuevi"

display(
    "Total Revenue Vehicles, from NTD",
    diff_voms_ref_veh[
        (diff_voms_ref_veh["agency_name"].str.contains(agency))
        | (diff_voms_ref_veh["agency_mart"].str.contains(agency))
    ],
    "",
    "Total Routes, from GTFS roll up and bridge table",
    rollup_route_bridge[
        (rollup_route_bridge["schedule_gtfs_dataset_name"].str.contains(agency))
        | (rollup_route_bridge["analysis_name"].str.contains(agency))
        | (rollup_route_bridge["schedule_name"].str.contains(agency))
    ],
    "",
    "Merged Revenue vehicles to total routes",
    rev_veh_route[rev_veh_route["agency_name"].str.contains(agency)],
    "",
    "List of agencies without GTFS-RT, CS team",
    no_rt[
        (no_rt["organization_name"].str.contains(agency))
        | (no_rt["service_name"].str.contains(agency))
        ],
    "",
    "merged data revenue vehicles x total routes to no gtfs_rt agencies",
    clean_check[
        (clean_check["agency_name"].str.contains(agency))
        | (clean_check["schedule_gtfs_dataset_name"].str.contains(agency))
        | (clean_check["analysis_name"].str.contains(agency))
        | (clean_check["service_name"].str.contains(agency))
    ],
    "",
    "Merged data, outer join",
    clean_check_outer_merge[
        (clean_check_outer_merge["agency_name"].str.contains(agency))
        | (clean_check_outer_merge["schedule_gtfs_dataset_name"].str.contains(agency))
        | (clean_check_outer_merge["analysis_name"].str.contains(agency))
        | (clean_check_outer_merge["service_name"].str.contains(agency))
    ],
)

'Total Revenue Vehicles, from NTD'

,Unnamed: 0,ntd_id,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service,ntd_id_mart,agency_mart,agency_voms_mart,report_year_mart,voms_diff,total_fleet_veh_voms_diff
200,200,99316,Chemehuevi Indian Tribe,2,2,2.0,99316,Chemehuevi Indian Tribe,2.0,2024,0.0,0.0


''

'Total Routes, from GTFS roll up and bridge table'

,schedule_gtfs_dataset_name,analysis_name,ntd_id_2022,schedule_name,n_routes


...

'merged data revenue vehicles x total routes to no gtfs_rt agencies'

,Unnamed: 0,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service,schedule_gtfs_dataset_name,analysis_name,ntd_id_2022,n_routes,service_name,is_the_agency_pursuing_rt_(via_the_msas_or_not)?,notes,passiogo,gtfs_rt,if_gtfs-rt_data_-_add_the_link,_merge


''

'Merged data, outer join'

,Unnamed: 0,agency_name,total_fleet_vehicles,active_fleet_vehicles,modetos_vehicles_operated_in_maximum_service,schedule_gtfs_dataset_name,analysis_name,ntd_id_2022,n_routes,service_name,is_the_agency_pursuing_rt_(via_the_msas_or_not)?,notes,passiogo,gtfs_rt,if_gtfs-rt_data_-_add_the_link,_merge
